In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# ติดตั้ง rpy2 (เชื่อมระหว่าง Python และ R)
!pip install rpy2

# โหลด extension เพื่อใช้ R ใน Jupyter/Colab
%load_ext rpy2.ipython


In [ ]:
%%R
# ติดตั้งแพ็กเกจ (รันแค่ครั้งเดียว)
install.packages("INLA", repos=c(getOption("repos"),
        INLA="https://inla.r-inla-download.org/R/stable"), dep=TRUE)
install.packages("spdep")
install.packages("sf")
install.packages("tidyverse")
install.packages("readxl")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘locfit’, ‘ash’, ‘FNN’, ‘kernlab’, ‘mclust’, ‘multicool’, ‘pracma’, ‘hdrcde’, ‘ks’, ‘bitops’, ‘rainbow’, ‘RCurl’, ‘pcaPP’, ‘fds’, ‘deSolve’, ‘leaps’, ‘inline’, ‘rrcov’, ‘raster’, ‘DEoptimR’, ‘RcppArmadillo’, ‘fda’, ‘tis’, ‘jpeg’, ‘BMA’, ‘rworldmap’, ‘dotCall64’, ‘rbibutils’, ‘wk’, ‘SparseM’, ‘proxy’, ‘tensorA’, ‘robustbase’, ‘bayesm’, ‘Ecfun’, ‘foreach’, ‘iterators’, ‘spam’, ‘maps’, ‘litedown’, ‘dfidx’, ‘Formula’, ‘zoo’, ‘lmtest’, ‘statmod’, ‘Rdpack’, ‘coda’, ‘classInt’, ‘s2’, ‘units’, ‘mnormt’, ‘quantreg’, ‘spData’, ‘e1071’, ‘fmesher’, ‘MatrixModels’, ‘compositions’, ‘Deriv’, ‘Ecdat’, ‘deldir’, ‘doParallel’, ‘evd’, ‘fastGHQuad’, ‘fields’, ‘gsl’, ‘gridExtra’, ‘markdown’, ‘matrixStats’, ‘mlogit’, ‘mvtnorm’, ‘numDeriv’, ‘pixmap’, ‘rgl’, ‘runjags’, ‘sf’, ‘sn’, ‘sp’, ‘spdep’, ‘splancs’, ‘terra’, ‘tidyterra’, ‘INLAspacetime’

trying URL 'https://cran.rstudio.com/src/contrib/loc

In [ ]:
%%R
library(INLA)
library(spdep)
library(sf)
library(tidyverse)
library(readxl)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ tidyr::expand() masks Matrix::expand()
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
✖ tidyr::pack()   masks Matrix::pack()
✖ tidyr::unpack() masks Matrix::unpack()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: Matrix
This is INLA_24.12.11 built 2024-12-11 19:58:26 UTC.
 - See www.r-inla.org/contact-us for how to get help.
 - List available models/likelihoods/etc with inla.list.models()
 - Use inla.doc(<NAME>) to access documentation
Loading required package: spData
To access larger datasets in this package, install the spDataLarge
package with: `install.packages('spDataLarge',
repos='https://nowosad.github.io/drat/', type='source')`
Loading required package: sf
Linking to GEOS 3.11.1, GDAL 3.6.4, PROJ 9.1.1; sf_use_s2() is TRUE


In [ ]:
%%R
# อ่านข้อมูล
data <- read_excel("/content/gdrive/MyDrive/Senior Project/Spatial/DATA/Klebsiella pneumoniae/Ceftazidime/PERCENT_R_Klebsiella pneumoniae_Ceftazidime.xlsx")
data

# A tibble: 1,392 × 6
   GROUP_NAME            ANTIBIOTIC  REGION SPEC_DATE  Year R_percent
   <chr>                 <chr>        <dbl> <chr>     <dbl>     <dbl>
 1 Klebsiella pneumoniae Ceftazidime      1 2015-01    2015      16.7
 2 Klebsiella pneumoniae Ceftazidime      2 2015-01    2015      75  
 3 Klebsiella pneumoniae Ceftazidime      3 2015-01    2015      43.8
 4 Klebsiella pneumoniae Ceftazidime      4 2015-01    2015      36.7
 5 Klebsiella pneumoniae Ceftazidime      5 2015-01    2015      25.5
 6 Klebsiella pneumoniae Ceftazidime      6 2015-01    2015      40.5
 7 Klebsiella pneumoniae Ceftazidime      7 2015-01    2015      43.6
 8 Klebsiella pneumoniae Ceftazidime      8 2015-01    2015      31.1
 9 Klebsiella pneumoniae Ceftazidime      9 2015-01    2015      36.1
10 Klebsiella pneumoniae Ceftazidime     10 2015-01    2015      32.4
# ℹ 1,382 more rows
# ℹ Use `print(n = ...)` to see more rows


In [ ]:
%%R
# เข้ารหัสพื้นที่ (province) และเวลา (month)
data$Region_id <- as.numeric(as.factor(data$REGION))
# สร้าง factor แล้วแปลงเป็นเลขที่เรียง
data$month_id <- as.numeric(as.factor(data$SPEC_DATE))

data

# A tibble: 1,392 × 8
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <dbl>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 1,

In [ ]:
%%R
unique(data$month_id)

  [1]   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
 [19]  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
 [37]  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
 [55]  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
 [73]  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
 [91]  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108


In [ ]:
%%R
library(sf)
library(spdep)

# โหลด shapefile ของจังหวัด
shp <- st_read("/content/gdrive/MyDrive/Senior Project/Spatial/Regions_no_province_boundaries.json")  # สมมุติว่าคุณมี shapefile แล้ว

# สร้าง adjacency matrix
nb <- poly2nb(shp)
nb2INLA("adjacency.graph", nb)

Reading layer `Regions_no_province_boundaries' from data source 
  `/content/gdrive/.shortcut-targets-by-id/1h5EYQx4w-VInRE_mIX08fjO5fl1RpQIz/Senior Project/Spatial/Regions_no_province_boundaries.json' 
  using driver `GeoJSON'
Simple feature collection with 13 features and 10 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 97.34336 ymin: 5.613038 xmax: 105.637 ymax: 20.46503
Geodetic CRS:  WGS 84


In [ ]:
%%R
shp

Simple feature collection with 13 features and 10 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: 97.34336 ymin: 5.613038 xmax: 105.637 ymax: 20.46503
Geodetic CRS:  WGS 84
First 10 features:
   HealthRegion   id           ADM1_EN    ADM1_TH  ADM0_EN   ADM0_TH ADM0_PCODE
1             1 <NA>        Chiang Mai    เชียงใหม่ Thailand ประเทศไทย         TH
2             2 <NA>         Uttaradit     อุตรดิตถ์ Thailand ประเทศไทย         TH
3             3 <NA>          Chai Nat      ชัยนาท Thailand ประเทศไทย         TH
4             4 <NA>        Nonthaburi      นนทบุรี Thailand ประเทศไทย         TH
5             5 <NA>        Ratchaburi      ราชบุรี Thailand ประเทศไทย         TH
6             6 <NA>      Samut Prakan สมุทรปราการ Thailand ประเทศไทย         TH
7             7 <NA>         Khon Kaen     ขอนแก่น Thailand ประเทศไทย         TH
8             8 <NA>         Bueng Kan      บึงกาฬ Thailand ประเทศไทย         TH
9             9 <NA> Nakhon Ratchasima  นครราชสีม

In [ ]:
%%R
# แมปข้อมูล %R ของแต่ละจังหวัดใส่ shapefile
shp$HealthRegion_id	 <- as.numeric(as.factor(shp$HealthRegion))
data$Region_id <- match(data$REGION, shp$HealthRegion)

In [ ]:
%%R
data

# A tibble: 1,392 × 8
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 1,

In [ ]:
%%R
unique(data$Region_id)

 [1]  1  2  3  4  5  6  7  8  9 10 11 12 13


In [ ]:
%%R
library(lubridate)

# ดึงเดือนออกมาจาก SPEC_DATE
data$month_num <- month(as.Date(paste0(data$SPEC_DATE, "-01")))

# สร้างตัวแปร sine/cosine สำหรับ seasonal effect
data$sin_month <- sin(2 * pi * data$month_num / 12)
data$cos_month <- cos(2 * pi * data$month_num / 12)


In [ ]:
%%R
library(lubridate)

# ดึงเดือนจาก SPEC_DATE
data$month <- month(as.Date(paste0(data$SPEC_DATE, "-01")))

# แบ่งฤดูกาลแบบไทย (ตัวอย่าง: 3 ฤดู)
data$season <- case_when(
  data$month %in% c(3, 4, 5) ~ "summer",   # มีนาคม–พฤษภาคม
  data$month %in% c(6, 7, 8, 9) ~ "rainy",  # มิถุนายน–กันยายน
  data$month %in% c(10, 11, 12, 1, 2) ~ "winter"  # ต.ค.–ก.พ.
)

# แปลงเป็นรหัสตัวเลข (ID)
data$season_id <- as.numeric(as.factor(data$season))


In [ ]:
%%R
data$region_year <- interaction(data$Region_id, data$Year)

In [ ]:
%%R
data$R_scaled <- (data$R_percent + 0.001) / 100

In [ ]:
%%R
data$R_scaled <- pmin(pmax(data$R_scaled, 0.0001), 0.9999)  # ตัดค่าขอบ

In [ ]:
%%R
data$R_logit <- log(data$R_scaled / (1 - data$R_scaled))  # หรือ: logit(x)

In [ ]:
%%R
# แบ่งชุดข้อมูล
data$set <- ifelse(data$Year %in% 2022:2023, "test", "train")
data

# A tibble: 1,392 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 1

In [ ]:
%%R
train_data <- data[data$set == "train", ]
test_data <- data[data$set == "test", ]
train_data

# A tibble: 1,080 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 1

In [ ]:
%%R
test_data$R_logit_real <- test_data$R_logit# เก็บของจริงไว้ใช้เทียบภายหลัง
test_data$R_logit <- NA  # ให้โมเดลพยากรณ์
test_data$R_logit_real

  [1] -0.392844763 -0.980778837 -0.393228269 -0.047027489 -0.235923680
  [6] -0.087928696 -0.018479044 -0.659579978 -0.243189715  0.336513380
 [11] -0.360400114 -0.151975976 -0.686413293 -0.723873366 -1.945818724
 [16] -0.521254144 -0.798460974 -0.479530736 -0.093778667 -0.204260045
 [21] -0.408926043 -0.206573821  0.473560856 -0.713169438 -0.116684138
 [26] -0.647293306 -0.670113001 -0.194115636 -0.746447325 -0.469961379
 [31] -0.625384966 -0.439324698 -0.171231427 -0.462581344 -0.273427883
 [36]  0.271616356 -0.432091459 -0.460773042 -1.008612986 -0.714155269
 [41] -0.971810371 -0.529651236 -0.125122986 -0.639614263 -0.368866136
 [46]  0.222053464 -0.430203257 -0.384178505  0.319635229 -0.266587948
 [51] -0.215934594 -1.013038944 -0.764926773 -1.303996708 -0.499354382
 [56]  0.070149615 -0.514316492 -0.331736282  0.037984946 -0.852729603
 [61] -0.164509116  0.425709655 -0.456195316 -0.173052450 -1.002101111
 [66] -0.686457737 -0.532761624 -0.485465412 -0.112755367 -0.787353192
 [71] 

In [ ]:
%%R
# Add the R_scaled_real column to train_data and fill it with NAs
train_data$R_logit_real <- NA

# Now, combine the data frames
combined_data <- rbind(train_data, test_data)
combined_data

# A tibble: 1,392 × 19
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 1

In [ ]:
%%R
# จากนั้นใช้ formula แบบ besag ได้เลย
formula <- R_scaled ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")+ sin_month + cos_month + f(region_year, model="iid")
#formula <- R_scaled ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")
result <- inla(formula,
               family="beta",
               data=combined_data,
               control.predictor=list(compute=TRUE),
               control.compute=list(dic=TRUE),
               control.inla = list(strategy = "simplified.laplace"))

# ดูผลลัพธ์
summary(result)

Time used:
    Pre = 17.1, Running = 6.98, Post = 0.101, Total = 24.2 
Fixed effects:
              mean    sd 0.025quant 0.5quant 0.975quant   mode kld
(Intercept) -0.550 0.045     -0.641   -0.550     -0.458 -0.550   0
sin_month   -0.019 0.017     -0.053   -0.020      0.015 -0.020   0
cos_month    0.024 0.011      0.003    0.024      0.046  0.024   0

Random effects:
  Name	  Model
    Region_id BYM2 model
   month_id RW2 model
   Year RW1 model
   region_year IID model

Model hyperparameters:
                                                  mean       sd 0.025quant
precision parameter for the beta observations 6.38e+01 2.52e+00   5.89e+01
Precision for Region_id                       4.09e+01 2.36e+01   1.29e+01
Phi for Region_id                             4.11e-01 2.76e-01   2.10e-02
Precision for month_id                        4.46e+04 2.65e+04   1.19e+04
Precision for Year                            3.91e+01 3.08e+01   8.77e+00
Precision for region_year                     1.48

In [ ]:
%%R
combined_data$predicted_scaled <- result$summary.fitted.values$mean
test_data$predicted_scaled <- combined_data$predicted_scaled[combined_data$set == "test"]

In [ ]:
%%R
test_data$predicted_percent <- (test_data$predicted_scaled * 100) - 0.001
test_data$actual_percent    <- test_data$R_percent

In [ ]:
%%R
test_data$actual_percent

  [1] 40.30227 27.27273 40.29304 48.82353 44.12811 47.80220 49.53704 34.08240
  [9] 43.94904 58.33333 41.08527 46.20690 33.48214 32.65306 12.50000 37.25490
 [17] 31.03448 38.23529 47.65625 44.91018 39.91597 44.85294 61.62162 32.88889
 [25] 47.08520 34.35897 33.84615 45.16129 32.15859 38.46154 34.85477 39.18919
 [33] 45.72864 38.63636 43.20557 56.74797 39.36170 38.67925 26.72414 32.86713
 [41] 27.45098 37.05882 46.87500 34.53237 40.88050 55.52764 39.40678 40.51095
 [49] 57.92254 43.37349 44.62151 26.63755 31.75676 21.34831 37.76824 51.75202
 [57] 37.41722 41.78082 50.94851 29.88506 45.89552 60.48387 38.78788 45.68345
 [65] 26.85185 33.48115 36.98630 38.09524 47.18310 31.27273 46.32353 53.20197
 [73] 38.84615 37.23849 57.86408 42.18750 43.07692 26.38889 32.55814 35.63218
 [81] 32.62411 35.03185 14.86486 47.34043 53.62319 32.16783 44.29530 59.08142
 [89] 36.99634 40.46921 23.67347 23.20442 16.66667 31.81818 41.37931 20.00000
 [97] 39.24051 48.78049 32.43243 52.07101 53.50195 40.69767 43.4

In [ ]:
%%R
test_data$predicted_percent

  [1] 34.62099 31.20249 39.70224 43.17276 32.28151 45.87433 50.97422 37.29837
  [9] 46.89165 58.37635 40.93483 44.45424 29.50462 33.96420 30.57992 39.00724
 [17] 42.46024 31.64731 45.15301 50.24784 36.61958 46.16812 57.66954 40.23281
 [25] 43.73702 28.90138 33.34175 29.99124 38.34634 41.78112 31.04718 44.46425
 [33] 49.55189 35.97512 45.47678 56.98897 39.56468 43.05282 28.33162 32.83231
 [41] 29.51034 37.80394 41.22270 30.55662 43.89707 48.97717 35.44692 44.90715
 [49] 56.42464 39.01599 42.48984 27.86660 32.47959 29.17781 37.42766 40.83479
 [57] 30.21724 43.50265 48.57671 35.08082 44.51088 56.03027 38.63516 42.09855
 [65] 27.54526 32.28928 28.99855 37.22442 40.62509 30.03421 43.28930 48.35983
 [73] 34.88319 44.29648 55.81632 38.42940 41.88697 27.37209 32.22604 28.93903
 [81] 37.15685 40.55530 29.97337 43.21828 48.28757 34.81748 44.22509 55.74494
 [89] 38.36095 41.81655 27.31459 32.22961 28.94245 37.16063 40.55916 29.97678
 [97] 43.22218 48.29148 34.82117 44.22901 55.74872 38.36476 41.8

In [ ]:
%%R
wape <- sum(abs(test_data$predicted_percent - test_data$actual_percent)) /
        sum(abs(test_data$actual_percent))

wape_percent <- round(wape * 100, 2)

cat("WAPE for 2022–2023 =", wape_percent, "%\n")


WAPE for 2022–2023 = 8.5 %


In [ ]:
%%R

test_data_valid <- test_data[test_data$actual_percent > 0, ]

# คำนวณ MAPE
mape_val <- mean(abs(test_data_valid$predicted_percent - test_data_valid$actual_percent) /
                 test_data_valid$actual_percent) * 100

# แสดงผล
cat("MAPE =", round(mape_val, 2), "%\n")


MAPE = 9.6 %


## Predict

In [ ]:
%%R
data

# A tibble: 1,392 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 1

In [ ]:
%%R
data

# A tibble: 1,392 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 1

In [ ]:
%%R
future_years <- 2024:2028
future_months <- 1:12
future_regions <- sort(unique(data$REGION))  # หรือ Health Region ที่มี
future_GROUP_NAME <- unique(data$GROUP_NAME)
future_ANTIBIOTIC <- unique(data$ANTIBIOTIC)
# สร้าง grid ของข้อมูลล่วงหน้า
future_data <- expand.grid(
  GROUP_NAME = future_GROUP_NAME,
  ANTIBIOTIC = future_ANTIBIOTIC,
  REGION = future_regions,
  month = future_months,
  Year = future_years
)
future_data

               GROUP_NAME  ANTIBIOTIC REGION month Year
1   Klebsiella pneumoniae Ceftazidime      1     1 2024
2   Klebsiella pneumoniae Ceftazidime      2     1 2024
3   Klebsiella pneumoniae Ceftazidime      3     1 2024
4   Klebsiella pneumoniae Ceftazidime      4     1 2024
5   Klebsiella pneumoniae Ceftazidime      5     1 2024
6   Klebsiella pneumoniae Ceftazidime      6     1 2024
7   Klebsiella pneumoniae Ceftazidime      7     1 2024
8   Klebsiella pneumoniae Ceftazidime      8     1 2024
9   Klebsiella pneumoniae Ceftazidime      9     1 2024
10  Klebsiella pneumoniae Ceftazidime     10     1 2024
11  Klebsiella pneumoniae Ceftazidime     11     1 2024
12  Klebsiella pneumoniae Ceftazidime     12     1 2024
13  Klebsiella pneumoniae Ceftazidime     13     1 2024
14  Klebsiella pneumoniae Ceftazidime      1     2 2024
15  Klebsiella pneumoniae Ceftazidime      2     2 2024
16  Klebsiella pneumoniae Ceftazidime      3     2 2024
17  Klebsiella pneumoniae Ceftazidime      4    

In [ ]:
%%R
library(dplyr)

future_data <- future_data %>%
  mutate(SPEC_DATE = sprintf("%d-%02d", Year, month))
future_data

               GROUP_NAME  ANTIBIOTIC REGION month Year SPEC_DATE
1   Klebsiella pneumoniae Ceftazidime      1     1 2024   2024-01
2   Klebsiella pneumoniae Ceftazidime      2     1 2024   2024-01
3   Klebsiella pneumoniae Ceftazidime      3     1 2024   2024-01
4   Klebsiella pneumoniae Ceftazidime      4     1 2024   2024-01
5   Klebsiella pneumoniae Ceftazidime      5     1 2024   2024-01
6   Klebsiella pneumoniae Ceftazidime      6     1 2024   2024-01
7   Klebsiella pneumoniae Ceftazidime      7     1 2024   2024-01
8   Klebsiella pneumoniae Ceftazidime      8     1 2024   2024-01
9   Klebsiella pneumoniae Ceftazidime      9     1 2024   2024-01
10  Klebsiella pneumoniae Ceftazidime     10     1 2024   2024-01
11  Klebsiella pneumoniae Ceftazidime     11     1 2024   2024-01
12  Klebsiella pneumoniae Ceftazidime     12     1 2024   2024-01
13  Klebsiella pneumoniae Ceftazidime     13     1 2024   2024-01
14  Klebsiella pneumoniae Ceftazidime      1     2 2024   2024-02
15  Klebsi

In [ ]:
%%R
# เข้ารหัสพื้นที่ (province) และเวลา (month)
future_data$Region_id <- as.numeric(as.factor(future_data$REGION))
# สร้าง factor แล้วแปลงเป็นเลขที่เรียง
future_data$month_id <- as.numeric(as.factor(future_data$SPEC_DATE))

# สร้าง region-year interaction (ถ้าใช้ในโมเดล)
future_data$region_year <- interaction(future_data$Region_id, future_data$Year)

# เติมค่าที่ยังไม่มี (target)
future_data$R_logit <- NA  # หรือ R_scaled หรือ R_percent ตามที่ใช้
future_data$R_scaled <- NA
future_data$R_percent<- NA
head(future_data,10)

              GROUP_NAME  ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Klebsiella pneumoniae Ceftazidime      1     1 2024   2024-01         1
2  Klebsiella pneumoniae Ceftazidime      2     1 2024   2024-01         2
3  Klebsiella pneumoniae Ceftazidime      3     1 2024   2024-01         3
4  Klebsiella pneumoniae Ceftazidime      4     1 2024   2024-01         4
5  Klebsiella pneumoniae Ceftazidime      5     1 2024   2024-01         5
6  Klebsiella pneumoniae Ceftazidime      6     1 2024   2024-01         6
7  Klebsiella pneumoniae Ceftazidime      7     1 2024   2024-01         7
8  Klebsiella pneumoniae Ceftazidime      8     1 2024   2024-01         8
9  Klebsiella pneumoniae Ceftazidime      9     1 2024   2024-01         9
10 Klebsiella pneumoniae Ceftazidime     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent
1         1      1.2024      NA       NA        NA
2         1      2.2024      NA       NA        NA
3         1      3.202

In [ ]:
%%R
library(lubridate)

# ดึงเดือนออกมาจาก SPEC_DATE
future_data$month_num <- month(as.Date(paste0(future_data$SPEC_DATE, "-01")))

# สร้างตัวแปร sine/cosine สำหรับ seasonal effect
future_data$sin_month <- sin(2 * pi * future_data$month_num / 12)
future_data$cos_month <- cos(2 * pi * future_data$month_num / 12)
head(future_data,10)

              GROUP_NAME  ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Klebsiella pneumoniae Ceftazidime      1     1 2024   2024-01         1
2  Klebsiella pneumoniae Ceftazidime      2     1 2024   2024-01         2
3  Klebsiella pneumoniae Ceftazidime      3     1 2024   2024-01         3
4  Klebsiella pneumoniae Ceftazidime      4     1 2024   2024-01         4
5  Klebsiella pneumoniae Ceftazidime      5     1 2024   2024-01         5
6  Klebsiella pneumoniae Ceftazidime      6     1 2024   2024-01         6
7  Klebsiella pneumoniae Ceftazidime      7     1 2024   2024-01         7
8  Klebsiella pneumoniae Ceftazidime      8     1 2024   2024-01         8
9  Klebsiella pneumoniae Ceftazidime      9     1 2024   2024-01         9
10 Klebsiella pneumoniae Ceftazidime     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.2024      NA  

In [ ]:
%%R
library(lubridate)

# ดึงเดือนจาก SPEC_DATE
future_data$month <- month(as.Date(paste0(future_data$SPEC_DATE, "-01")))

# แบ่งฤดูกาลแบบไทย (ตัวอย่าง: 3 ฤดู)
future_data$season <- case_when(
  future_data$month %in% c(3, 4, 5) ~ "summer",   # มีนาคม–พฤษภาคม
  future_data$month %in% c(6, 7, 8, 9) ~ "rainy",  # มิถุนายน–กันยายน
  future_data$month %in% c(10, 11, 12, 1, 2) ~ "winter"  # ต.ค.–ก.พ.
)

# แปลงเป็นรหัสตัวเลข (ID)
future_data$season_id <- as.numeric(as.factor(future_data$season))
head(future_data,10)

              GROUP_NAME  ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Klebsiella pneumoniae Ceftazidime      1     1 2024   2024-01         1
2  Klebsiella pneumoniae Ceftazidime      2     1 2024   2024-01         2
3  Klebsiella pneumoniae Ceftazidime      3     1 2024   2024-01         3
4  Klebsiella pneumoniae Ceftazidime      4     1 2024   2024-01         4
5  Klebsiella pneumoniae Ceftazidime      5     1 2024   2024-01         5
6  Klebsiella pneumoniae Ceftazidime      6     1 2024   2024-01         6
7  Klebsiella pneumoniae Ceftazidime      7     1 2024   2024-01         7
8  Klebsiella pneumoniae Ceftazidime      8     1 2024   2024-01         8
9  Klebsiella pneumoniae Ceftazidime      9     1 2024   2024-01         9
10 Klebsiella pneumoniae Ceftazidime     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.2024      NA  

In [ ]:
%%R
future_data$set <- NA
head(future_data,10)

              GROUP_NAME  ANTIBIOTIC REGION month Year SPEC_DATE Region_id
1  Klebsiella pneumoniae Ceftazidime      1     1 2024   2024-01         1
2  Klebsiella pneumoniae Ceftazidime      2     1 2024   2024-01         2
3  Klebsiella pneumoniae Ceftazidime      3     1 2024   2024-01         3
4  Klebsiella pneumoniae Ceftazidime      4     1 2024   2024-01         4
5  Klebsiella pneumoniae Ceftazidime      5     1 2024   2024-01         5
6  Klebsiella pneumoniae Ceftazidime      6     1 2024   2024-01         6
7  Klebsiella pneumoniae Ceftazidime      7     1 2024   2024-01         7
8  Klebsiella pneumoniae Ceftazidime      8     1 2024   2024-01         8
9  Klebsiella pneumoniae Ceftazidime      9     1 2024   2024-01         9
10 Klebsiella pneumoniae Ceftazidime     10     1 2024   2024-01        10
   month_id region_year R_logit R_scaled R_percent month_num sin_month
1         1      1.2024      NA       NA        NA         1       0.5
2         1      2.2024      NA  

In [ ]:
%%R
head(data,10)

# A tibble: 10 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <int>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 10 m

In [ ]:
%%R
combined_data_future <- rbind(data, future_data)
combined_data_future

# A tibble: 2,172 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <dbl>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 2

In [ ]:
%%R
combined_data_future$R_scaled <- (combined_data_future$R_percent + 0.001) / 100
combined_data_future

# A tibble: 2,172 × 18
   GROUP_NAME     ANTIBIOTIC REGION SPEC_DATE  Year R_percent Region_id month_id
   <chr>          <chr>       <dbl> <chr>     <dbl>     <dbl>     <dbl>    <dbl>
 1 Klebsiella pn… Ceftazidi…      1 2015-01    2015      16.7         1        1
 2 Klebsiella pn… Ceftazidi…      2 2015-01    2015      75           2        1
 3 Klebsiella pn… Ceftazidi…      3 2015-01    2015      43.8         3        1
 4 Klebsiella pn… Ceftazidi…      4 2015-01    2015      36.7         4        1
 5 Klebsiella pn… Ceftazidi…      5 2015-01    2015      25.5         5        1
 6 Klebsiella pn… Ceftazidi…      6 2015-01    2015      40.5         6        1
 7 Klebsiella pn… Ceftazidi…      7 2015-01    2015      43.6         7        1
 8 Klebsiella pn… Ceftazidi…      8 2015-01    2015      31.1         8        1
 9 Klebsiella pn… Ceftazidi…      9 2015-01    2015      36.1         9        1
10 Klebsiella pn… Ceftazidi…     10 2015-01    2015      32.4        10        1
# ℹ 2

In [ ]:
%%R
combined_data_future$R_scaled

   [1] 0.16667667 0.75001000 0.43751000 0.36654386 0.25499008 0.40507329
   [7] 0.43590744 0.31074446 0.36083474 0.32353941 0.37689442 0.43530412
  [13] 0.26316789 0.36364636 0.60001000 0.35295118 0.30981392 0.24348826
  [19] 0.43056556 0.36765706 0.40001000 0.29762905 0.19231769 0.36843105
  [25] 0.39063500 0.17648059 0.28572429 0.77778778 0.12501000 0.26733673
  [31] 0.27341824 0.27849101 0.40299507 0.30657934 0.28283828 0.20001000
  [37] 0.35752295 0.35136135 0.27028027 0.26924077 0.81819182 0.33334333
  [43] 0.27028027 0.22747781 0.38462538 0.41989950 0.36943675 0.27473527
  [49] 0.14286714 0.31251000 0.32001000 0.31859407 0.10001000 1.00001000
  [55] 0.33334333 0.30040526 0.22120816 0.41490362 0.42858143 0.30460770
  [61] 0.31326301 0.15790474 0.34839710 0.31035483 0.35295118 0.33334333
  [67] 0.57143857 0.46667667 0.34229188 0.23900371 0.40187916 0.41090109
  [73] 0.28001000 0.31765706 0.30435783 0.34181791 0.31251000 0.28126000
  [79] 0.26667667 0.56411256 0.07143857 0.29693833 

In [ ]:
%%R
# จากนั้นใช้ formula แบบ besag ได้เลย
formula <- R_scaled ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")+ sin_month + cos_month + f(region_year, model="iid")
#formula <- R_scaled ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")
result <- inla(formula,
               family="beta",
               data=combined_data_future,
               control.predictor=list(compute=TRUE),
               control.compute=list(dic=TRUE),
               control.inla = list(strategy = "simplified.laplace"))

# ดูผลลัพธ์
summary(result)


 *** inla.core.safe:  The inla program failed, but will rerun in case better initial values may help. try=1/1 
Error in inla.core.safe(formula = formula, family = family, contrasts = contrasts,  : 
  The inla-program exited with an error. Unless you interupted it yourself, please rerun with verbose=TRUE and check the output carefully.
  If this does not help, please contact the developers at <help@r-inla.org>.
The inla program failed and the maximum number of tries has been reached.


RInterpreterError: Failed to parse and evaluate line '# จากนั้นใช้ formula แบบ besag ได้เลย\nformula <- R_scaled ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")+ sin_month + cos_month + f(region_year, model="iid")\n#formula <- R_scaled ~ 1  + f(Region_id, model="bym2", graph="adjacency.graph") + f(month_id, model="rw2")+ f(Year, model="rw1")\nresult <- inla(formula,\n               family="beta",\n               data=combined_data_future,\n               control.predictor=list(compute=TRUE),\n               control.compute=list(dic=TRUE),\n               control.inla = list(strategy = "simplified.laplace"))\n\n# ดูผลลัพธ์\nsummary(result)\n'.
R error message: 'Error in inla.core.safe(formula = formula, family = family, contrasts = contrasts,  : \n  The inla-program exited with an error. Unless you interupted it yourself, please rerun with verbose=TRUE and check the output carefully.\n  If this does not help, please contact the

In [ ]:
%%R
#ถ้า future_data$predicted_scaled ไม่มีติดลบใช้อันนี้
# คำนวณจำนวนแถว
#n_total       <- nrow(combined_data_future)
#n_future      <- nrow(future_data)
#start_index   <- n_total - n_future + 1
#end_index     <- n_total

# ดึงค่าที่พยากรณ์เฉพาะ future
#future_data$predicted_scaled <- result$summary.fitted.values$mean[start_index:end_index]
#future_data$predicted_scaled

In [ ]:
%%R
#ถ้า future_data$predicted_scaled มีติดลบใช้อันนี้
# คำนวณจำนวนแถว
n_total       <- nrow(combined_data_future)
n_future      <- nrow(future_data)
start_index   <- n_total - n_future + 1
end_index     <- n_total

# ดึงค่าที่พยากรณ์เฉพาะ future
inv_logit <- function(x) exp(x) / (1 + exp(x))
future_data$predicted_scaled <- inv_logit(result$summary.fitted.values$mean[start_index:end_index])
future_data$predicted_scaled

In [ ]:
%%R
# scale เป็น %
future_data$predicted_percent <- (future_data$predicted_scaled * 100) - 0.001

In [ ]:
%%R
head(future_data[, c("REGION", "SPEC_DATE", "predicted_percent")],13)

In [ ]:
%%R
install.packages("writexl")  # รันครั้งเดียว
library(writexl)

In [ ]:
%%R
write_xlsx(future_data, path = "/content/gdrive/MyDrive/Senior Project/Spatial/Spatiotemporal/results/Klebsiella pneumoniae/Ceftazidime/Future_R_Klebsiella pneumoniae_Ceftazidime.xlsx")